# Elo Modeling

Tujuan : 

Notebook ini bertujuan mengevaluasi pengaruh penambahan **Elo Rating** terhadap performa model dalam memprediksi hasil pertandingan sepak bola internasional.

Berbeda dengan *Historical Modeling* yang hanya memanfaatkan fitur dasar dan statistik historis, tahap ini menambahkan representasi kekuatan relatif tim melalui **Elo Rating** yang dihitung secara kronologis menggunakan seluruh riwayat pertandingan.

Eksperimen tetap menggunakan algoritma dan strategi pemodelan yang sama, yaitu **Logistic Regression** dan **Random Forest**, sehingga setiap perubahan performa dapat diatribusikan pada informasi tambahan yang diberikan oleh fitur Elo, bukan karena perubahan model atau proses *preprocessing*.

Hasil evaluasi pada notebook ini akan dibandingkan dengan eksperimen sebelumnya untuk mengetahui apakah Elo Rating mampu meningkatkan kualitas prediksi secara terukur.

## 1. Load Elo Dataset

Dataset yang digunakan pada tahap ini merupakan hasil dari notebook **Elo Rating Engine**. Dataset tersebut telah memuat seluruh fitur dasar (*baseline features*), fitur historis (*historical features*), serta fitur **Elo Rating** yang dibangun secara kronologis.
Sebelum memulai proses pemodelan, kita akan memuat dataset dan melakukan pengecekan singkat untuk memastikan struktur data sesuai dengan yang diharapkan.

In [12]:
import pandas as pd

In [13]:
elo_df = pd.read_csv("../data/processed/elo_features.csv")

In [14]:
elo_df.head()

,date,home_team,away_team,home_score,away_score,tournament,neutral,home_last5_winrate,away_last5_winrate,home_last10_winrate,...,away_avg_goals_scored,home_avg_goals_conceded,away_avg_goals_conceded,home_goal_difference_form,away_goal_difference_form,match_result,home_elo_before,away_elo_before,home_elo_after,away_elo_after
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,False,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,D,1500.000000,1500.000000,1500.000000,1500.000000
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,False,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,H,1500.000000,1500.000000,1510.000000,1490.000000
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,False,0.000000,0.500000,0.000000,...,2.000000,2.000000,1.000000,-1.000000,1.000000,H,1490.000000,1510.000000,1500.575011,1499.424989
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,False,0.333333,0.333333,0.333333,...,1.333333,1.333333,1.666667,0.333333,-0.333333,D,1499.424989,1500.575011,1499.458089,1500.541911
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,False,0.250000,0.250000,0.250000,...,1.750000,1.750000,1.500000,-0.250000,0.250000,H,1500.541911,1499.458089,1510.510716,1489.489284


In [15]:
elo_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 49433 entries, 0 to 49432
Data columns (total 22 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   date                       49433 non-null  str    
 1   home_team                  49433 non-null  str    
 2   away_team                  49433 non-null  str    
 3   home_score                 49433 non-null  float64
 4   away_score                 49433 non-null  float64
 5   tournament                 49433 non-null  str    
 6   neutral                    49433 non-null  bool   
 7   home_last5_winrate         49292 non-null  float64
 8   away_last5_winrate         49238 non-null  float64
 9   home_last10_winrate        49292 non-null  float64
 10  away_last10_winrate        49238 non-null  float64
 11  home_avg_goals_scored      49292 non-null  float64
 12  away_avg_goals_scored      49238 non-null  float64
 13  home_avg_goals_conceded    49292 non-null  float64
 14  a

Dataset berhasil dimuat dan telah memuat seluruh fitur yang dibutuhkan untuk proses pemodelan. Selain fitur dasar dan fitur historis, dataset kini juga memiliki empat kolom Elo Rating, yaitu:

- `home_elo_before`
- `away_elo_before`
- `home_elo_after`
- `away_elo_after`

Pada tahap pemodelan, hanya **Elo sebelum pertandingan** (`home_elo_before` dan `away_elo_before`) yang akan digunakan sebagai fitur prediktif karena nilai tersebut memang tersedia sebelum pertandingan dimulai. Sebaliknya, `home_elo_after` dan `away_elo_after` merupakan hasil pembaruan setelah pertandingan selesai sehingga tidak digunakan sebagai input model untuk menghindari *future data leakage*.

## 3. Feature & Target Separation

Pada tahap ini, kita akan memisahkan **fitur (X)** dan **target (y)** yang akan digunakan pada proses pemodelan.
Variabel target tetap menggunakan `match_result` yang dibentuk berdasarkan hasil akhir pertandingan. Sementara itu, fitur yang digunakan merupakan kombinasi dari:
- **Baseline Features**
- **Historical Features**
- **Elo Rating Features**
Seluruh fitur dipilih berdasarkan informasi yang memang tersedia **sebelum pertandingan dimulai**, sehingga proses pemodelan tetap terhindar dari *future data leakage*.

In [16]:
# MEMBENTUK TARGET
def get_match_result(row):
    if row["home_score"] > row["away_score"]:
        return "H"
    elif row["home_score"] < row["away_score"]:
        return "A"
    else:
        return "D"

elo_df["match_result"] = elo_df.apply(get_match_result, axis=1)

In [17]:
# MENENTUKAN FEATURE
feature_columns = [

    # Baseline Features
    "home_team",
    "away_team",
    "tournament",
    "neutral",

    # Historical Features
    "home_last5_winrate",
    "away_last5_winrate",
    "home_last10_winrate",
    "away_last10_winrate",
    "home_avg_goals_scored",
    "away_avg_goals_scored",
    "home_avg_goals_conceded",
    "away_avg_goals_conceded",
    "home_goal_difference_form",
    "away_goal_difference_form",

    # Elo Features
    "home_elo_before",
    "away_elo_before"
]

In [18]:
# PISAHKAN X DAN Y
X = elo_df[feature_columns]

y = elo_df["match_result"]

In [19]:
# CEK BENTUK
print(f"Feature Shape : {X.shape}")
print(f"Target Shape  : {y.shape}")

Feature Shape : (49433, 16)
Target Shape  : (49433,)
